###  在Agent运行图中访问长期记忆
我们可以在工具或中间件中访问长期记忆。
#### 1. 在工具中访问长期记忆
1.1 基于InMemoryStore

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

out_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)


In [5]:
from langchain.messages import HumanMessage
from typing import NotRequired
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

class CustomState(AgentState):
    user_id: NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name: str, runtime: ToolRuntime) -> str:
    """
    将用户信息保存在长期记忆中

    Args:
        name: 用户名
        runtime: 运行时间

    Returns:
        str: 保存状态

    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    value = {"name": name}
    runtime.store.put(namespace, key, value)
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime: ToolRuntime) -> str:
    """
    从长期记忆中读取用户信息

    Args:
        runtime: 运行时间

    Returns:
        str: 用户信息
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    item = runtime.store.get(namespace, key)
    return str(item.value) if item else "unknown"

my_agent = create_agent(
    model=out_model,
    tools=[
        save_user_info,
        get_user_info,
    ],
    store=store,
    system_prompt="用户体积个人信息时及时记录，用户询问个人信息时尝试用工具检索",
    state_schema=CustomState,
)

print("=" * 30, '-> 第一个会话（线程） <-', "=" * 30)
response1 = my_agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
    "user_id": "user-1"
})
for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
response2 = my_agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1"
})
for msg in response2["messages"]:
    msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================
Tool Calls:
  save_user_info (call_8a811f0cfb384806bbeda19b)
 Call ID: call_8a811f0cfb384806bbeda19b
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

你好，小花！很高兴认识你。有什么我可以帮你的吗？
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_273646c2233541a98575eea7)
 Call ID: call_273646c2233541a98575eea7
  Args:
================================= Tool Message ======

1.2 基于PostgresStore

In [7]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent, AgentState
from langchain.tools import ToolRuntime,tool
from typing import NotRequired
from langgraph.store.postgres import PostgresStore
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DB_IP = os.getenv("POSTGRE_DB_IP")
DB_USER = os.getenv("POSTGRE_DB_USER")
DB_PWD = os.getenv("POSTGRE_DB_PWD")

DB_URL = f"postgresql://{DB_USER}:{DB_PWD}@{DB_IP}:5432/langchain_memory?sslmode=disable"

class CustomPgState(AgentState):
    user_id: NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name: str, runtime: ToolRuntime) -> str:
    """
    将用户信息保存在长期记忆中

    Args:
        name: 用户名
        runtime: 运行时间

    Returns:
        str: 保存状态

    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    value = {"name": name}
    runtime.store.put(namespace, key, value)
    return "saved"

@tool(parse_docstring=True)
def get_user_info(runtime: ToolRuntime) -> str:
    """
    从长期记忆中读取用户信息

    Args:
        runtime: 运行时间

    Returns:
        str: 用户信息
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    item = runtime.store.get(namespace, key)
    return str(item.value) if item else "unknown"

with PostgresStore.from_conn_string(DB_URL) as pg_store:
    pg_store.setup()
    my_agent = create_agent(
        model=out_model,
        tools=[save_user_info, get_user_info],
        system_prompt="用户体积个人信息时及时记录，用户询问个人信息时尝试用工具检索",
        state_schema=CustomPgState,
        store=pg_store,
    )

    print("=" * 30, " --> 第一个会话（线程）<-- ", "=" * 30)
    resp1 = my_agent.invoke({
        "messages": [HumanMessage(content="你好，很高兴认识你，我是小花")],
        "user_id": "user-1"
    })
    for msg in resp1["messages"]:
        msg.pretty_print()

    print("=" * 30, '-> 第二个会话（线程） <-', "=" * 30)
    resp2 = my_agent.invoke({
        "messages": [HumanMessage(content="我是谁")],
        "user_id": "user-1"
    })
    for msg in resp2["messages"]:
        msg.pretty_print()


==============================  --> 第一个会话（线程）<--  ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================
Tool Calls:
  save_user_info (call_084101c0ee9c40c989fd753d)
 Call ID: call_084101c0ee9c40c989fd753d
  Args:
    name: 小花
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

你好，小花！很高兴认识你。请问有什么我可以帮你的吗？
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_c397896c5d204265ae0fde60)
 Call ID: call_c397896c5d204265ae0fde60
  Args:
================================= Tool Message =